In [ ]:
import spikeinterface as si
import spikeinterface.extractors as se

import numpy as np
from pipeline.utils import log_recording
from pathlib import Path
from loguru import logger

class Timestamps:


    def __init__(self, name: str, fs: float, t_start: float = 0.0):
        self.name = name
        self.fs = fs
        self.dt = 1 / fs
        
        self.global_timestamps = []
        self.intervals = []
        self.t_offset = t_start
        self.starting_states = []

    def update(self, local_ts: np.ndarray, t_end: float, starting_state = None) -> None:
        """Add new session to global timeline."""
        # Update global timestamps
        self.global_timestamps.append(local_ts + self.t_offset + self.dt)

        # Update intervals
        self.intervals.append((self.t_offset, self.t_offset + t_end))
        
        # Update offset for next segment
        self.t_offset += t_end

        if starting_state:
            self.starting_states.append(starting_state)
        
    def __repr__(self):
        return f"Timestamps(name='{self.name}', @ {self.fs:.1f} Hz)"

class ProbeRun:
    """ Represents a single probe in an OpenEphys session."""
    
    
    def __init__(self, run_path, name, stream_id):
        self.run_path       = run_path
        self.name           = name
        self.stream_id      = stream_id
        self._recording     = None

    def load_sync(self):
        """ Load timestamps for synchronization."""
        ts_files = list(Path(self.run_path).rglob('timestamps.npy'))
        return ts_files

    def __repr__(self):
        return f"ProbeRun(name={self.name}, stream_id={self.stream_id})"

    @property
    def recording(self):
        if self._recording is None:
            self._recording = se.read_openephys(self.run_path, stream_id=self.stream_id)
        return self._recording
    
class MultiProbeRun:
    """ Represents multiple probes in a single OpenEphys session ."""


    def __init__(self, run_path):
        logger.info(f"Initializing MultiProbeRun for path: {run_path}")
        self.run_path = run_path
        self.session_name = Path(run_path).name
        self._recording = None
        self._probes = {}

    def load_recordings(self):
        """ Load recordings for all probes matching the filter."""
        logger.info(f"Loading recordings from {self.run_path}")
        stream_names, stream_ids = se.get_neo_streams('openephysbinary', self.run_path)

        for stream_name, stream_id in zip(stream_names, stream_ids):
            logger.info(f"Found stream: {stream_name} with ID: {stream_id}")
            # Extract probe name (e.g., "OneBox-0.ProbeA" -> "ProbeA")
            probe_name = stream_name.split(".")[-1]
            if "SYNC" not in probe_name:
                probe = ProbeRun(self.run_path, stream_name, stream_id)
                self._probes[probe_name] = probe

    def concatenate(self, probe_name):
        """ Concatenate probe across recordings."""
        if probe_name not in self._probes:
            raise ValueError(f"Probe {probe_name} not found in this session.")
        recording = self._probes[probe_name].recording
        log_recording(recording, f"Probe_{probe_name}_Recording")
        return self._probes[probe_name].recording

    def __getitem__(self, probe_name):
        return self._probes[probe_name]

    def __repr__(self):
        return f"MultiProbeRun(run_path={self.run_path}, probes={list(self._probes.keys())})"

class Experiment:
    """ Represents an experiment with multiple sessions."""


    def __init__(self, recording_paths):
        self.sessions = [MultiProbeRun(path) for path in recording_paths]
        self._probes = {}

    def preprocess(self):
        """ Preprocess all sessions."""
        for session in self.sessions:
            logger.info(f"Preprocessing session at {session.run_path}")
            session.load_recordings()

    def concatenate(self, probe_filter, save_path=None):
        """ Concatenate recordings for each probe across sessions."""
        for probe_name in probe_filter:
            logger.info(f"Concatenating sessions for probe: {probe_name}")
            probe_recordings = [session[probe_name].recording for session in self.sessions]
            concatenated_recording = si.concatenate_recordings(probe_recordings)
            log_recording(concatenated_recording, f"Concatenated_Probe_{probe_name}")
            self._probes[probe_name] = concatenated_recording
        
        if save_path:
            for probe_name, recording in self._probes.items():
                si.write_binary_recording(recording, save_path / f"Concatenated_Probe_{probe_name}")
        return self._probes
    
    def __repr__(self):
        return f"Experiment(sessions={len(self.sessions)})"


In [74]:
from pipeline.utils import load_config
conf_path = Path('R:\\Basic_Sciences\\Phys\\SenzaiLab\\pipeline_output\\configs\\config_example.yaml')
config = load_config(conf_path)
config

2025-11-07 15:20:48.105 | SUCCESS  | pipeline.utils:load_config:66 - Loaded configuration from: R:\Basic_Sciences\Phys\SenzaiLab\pipeline_output\configs\config_example.yaml


{'session_name': 'AA001_Day2',
 'recording_paths': ['R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
  'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField'],
 'local_output': 'E:/pipeline_output',
 'remote_output': 'R:/Basic_Sciences/Phys/SenzaiLab/pipeline_output',
 'fs': 30000.0,
 'target_fs': 1250.0,
 'save_kwargs': {'n_jobs': 16,
  'chunk_duration': '2s',
  'progress_bar': True,
  'mp_context': 'spawn',
  'verbose': True,
  'overwrite': True}}

In [86]:
session1 = MultiProbeRun(config['recording_paths'][0])
session1.load_recordings()

2025-11-07 18:23:35.805 | INFO     | __main__:__init__:68 - Initializing MultiProbeRun for path: R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\VisualStimuli\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-07 18:23:35.806 | INFO     | __main__:load_recordings:76 - Loading recordings from R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\VisualStimuli\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-07 18:23:36.737 | INFO     | __main__:load_recordings:80 - Found stream: Record Node 103#OneBox-106.OneBox-ADC with ID: 0
2025-11-07 18:23:36.737 | INFO     | __main__:load_recordings:80 - Found stream: Record Node 103#OneBox-106.ProbeA with ID: 1
2025-11-07 18:23:36.738 | INFO     | __main__:load_recordings:80 - Found stream: Record Node 103#OneBox-106.ProbeB with ID: 2
2025-11-07 18:23:36.738 | INFO     | __main__:load_recordings:80 - Found stream: Record Node 103#OneBox-106.ProbeC with ID: 3
2025-11-07 18:23:36.738 | INFO     | __main__:load_recordings:80 - Found stream: Rec

In [88]:
session1['ProbeA'].load_sync()

[]

In [83]:
config['recording_paths']

['R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual',
 'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField',
 'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField',
 'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\OpenField_Homecage\\AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField']

In [ ]:
conf_path = Path('R:\\Basic_Sciences\\Phys\\SenzaiLab\\pipeline_output\\configs\\config_example.yaml')
config = load_config(conf_path)


exp = Experiment(recording_paths=config['recording_paths'])
exp.preprocess()

2025-11-07 11:04:45.610 | INFO     | __main__:__init__:37 - Initializing MultiProbeRun for path: R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\VisualStimuli\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual
2025-11-07 11:04:45.610 | INFO     | __main__:__init__:37 - Initializing MultiProbeRun for path: R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\OpenField_Homecage\AA001_2025-10-09_13-41-27_4Probe_RSC_ADn_RecOpenField
2025-11-07 11:04:45.611 | INFO     | __main__:__init__:37 - Initializing MultiProbeRun for path: R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\OpenField_Homecage\AA001_2025-10-09_14-22-26_4Probe_RSC_ADn_RecOpenField
2025-11-07 11:04:45.612 | INFO     | __main__:__init__:37 - Initializing MultiProbeRun for path: R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\OpenField_Homecage\AA001_2025-10-09_14-57-57_4Probe_RSC_ADn_RecOpenField
2025-11-07 11:04:45.612 | INFO     | __main__:preprocess:81 - Preprocessing session at R:\Basic_Sciences\Phys\SenzaiLab\Ayo\AA001\Day2\Visu

In [84]:
exp.concatenate(probe_filter=['ProbeA', 'ProbeB'])

2025-11-07 15:26:21.506 | INFO     | __main__:concatenate:87 - Concatenating sessions for probe: ProbeA
2025-11-07 15:26:21.508 | INFO     | pipeline.utils:log_recording:179 - Concatenated_Probe_ProbeA: 384 ch, 33692.2s (9.36 h) @ 30.0 kHz int16 (722.96 GB)
2025-11-07 15:26:21.508 | INFO     | __main__:concatenate:87 - Concatenating sessions for probe: ProbeB
2025-11-07 15:26:21.509 | INFO     | pipeline.utils:log_recording:179 - Concatenated_Probe_ProbeB: 384 ch, 34245.8s (9.51 h) @ 30.0 kHz int16 (734.83 GB)


{'ProbeA': ConcatenateSegmentRecording: 384 channels - 30.0kHz - 1 segments - 1,010,766,713 samples 
                              33,692.22s (9.36 hours) - int16 dtype - 722.96 GiB,
 'ProbeB': ConcatenateSegmentRecording: 384 channels - 30.0kHz - 1 segments - 1,027,373,501 samples 
                              34,245.78s (9.51 hours) - int16 dtype - 734.83 GiB}

In [70]:
exp._probes['ProbeA'].get_total_memory_size()

776268835584

In [71]:
from pipeline.utils import format_file_size
format_file_size(exp._probes['ProbeA'].get_total_memory_size())

'722.96 GB'

In [37]:
session = MultiProbeRun(run_path=config['recording_paths'][0])
session.load_recordings()

In [39]:
session['ProbeA'].run_path

'R:\\Basic_Sciences\\Phys\\SenzaiLab\\Ayo\\AA001\\Day2\\VisualStimuli\\AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual'

In [43]:
list(Path(session['ProbeA'].run_path).rglob('timestamps.npy'))

[WindowsPath('R:/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/MessageCenter/timestamps.npy'),
 WindowsPath('R:/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.ProbeC/TTL/timestamps.npy'),
 WindowsPath('R:/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.ProbeD/TTL/timestamps.npy'),
 WindowsPath('R:/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/events/OneBox-106.OneBox-ADC/TTL/timestamps.npy'),
 WindowsPath('R:/Basic_Sciences/Phys/SenzaiLab/Ayo/AA001/Day2/VisualStimuli/AA001_2025-10-09_12-31-37_4probe_RSC_ADn_Visual/Record Node 103/experiment1/recording1/e

In [32]:
session['ProbeA'].recording.get_probe()

Probe - 384ch - 4shanks